### Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import os
import pickle
import warnings

warnings.filterwarnings("ignore")

### General Function to be applied for both training & validation data.

In [2]:
categorical = ["PU_DO"]
numerical = ["trip_distance", "passenger_count"]

In [3]:
# To avoid Repetitive code
def load_data(file_path):
    # 1. Load the data
    df = pd.read_parquet(file_path)

    # 2. Calculate trip duration in minutes
    df["trip_duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    # 3. Filter out trips with duration less than 1 minute or greater than 60 minutes
    # Filter out trips with distance less than or equal to 0
    df = df[(df["trip_duration"] >= 1) & (df["trip_duration"] <= 60)]
    df = df[df["trip_distance"] > 0]

    # 4. Create categorical PU_DO feature
    categorincal_features = ["PULocationID", "DOLocationID"]
    df[categorincal_features] = df[categorincal_features].astype(str)
    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

    return df


# # Encode the 'store_and_fwd_flag' column to binary values
# df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({
#     'N': 0,
#     'Y': 1
# })

### Prepare Training Data

In [4]:
df_train = load_data("../data/raw/green_tripdata_2026-04.parquet")

print(f"Data Shape: {df_train.shape}")

df_train.head()

Data Shape: (41729, 23)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,trip_duration,PU_DO
0,2,2026-05-01 00:02:02,2026-05-01 00:21:03,N,1.0,198,56,1.0,4.08,21.90,...,0.0,NaN,1.0,24.40,2.0,1.0,0.00,0.0,19.016667,198_56
1,2,2026-05-01 00:30:22,2026-05-01 00:53:51,N,1.0,66,262,1.0,8.07,35.90,...,0.0,NaN,1.0,42.95,1.0,1.0,2.75,0.0,23.483333,66_262
2,2,2026-05-01 00:30:17,2026-05-01 00:50:34,N,1.0,66,112,1.0,5.97,27.50,...,0.0,NaN,1.0,31.00,1.0,1.0,0.00,0.0,20.283333,66_112
4,2,2026-05-01 00:45:47,2026-05-01 00:53:20,N,5.0,200,265,0.0,1.86,13.11,...,0.0,NaN,1.0,14.11,1.0,2.0,0.00,0.0,7.550000,200_265
8,2,2026-05-01 00:32:20,2026-05-01 00:42:36,N,1.0,95,160,1.0,2.25,13.50,...,0.0,NaN,1.0,16.00,1.0,1.0,0.00,0.0,10.266667,95_160


In [5]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 41729 entries, 0 to 44920
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               41729 non-null  int32         
 1   lpep_pickup_datetime   41729 non-null  datetime64[us]
 2   lpep_dropoff_datetime  41729 non-null  datetime64[us]
 3   store_and_fwd_flag     36752 non-null  str           
 4   RatecodeID             36752 non-null  float64       
 5   PULocationID           41729 non-null  str           
 6   DOLocationID           41729 non-null  str           
 7   passenger_count        36752 non-null  float64       
 8   trip_distance          41729 non-null  float64       
 9   fare_amount            41729 non-null  float64       
 10  extra                  41729 non-null  float64       
 11  mta_tax                41729 non-null  float64       
 12  tip_amount             41729 non-null  float64       
 13  tolls_amount     

In [6]:
# Check for missing values
df_train.isnull().sum()

VendorID                     0
lpep_pickup_datetime         0
lpep_dropoff_datetime        0
store_and_fwd_flag        4977
RatecodeID                4977
PULocationID                 0
DOLocationID                 0
passenger_count           4977
trip_distance                0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
ehail_fee                41729
improvement_surcharge        0
total_amount                 0
payment_type              4977
trip_type                 4977
congestion_surcharge      4977
cbd_congestion_fee           0
trip_duration                0
PU_DO                        0
dtype: int64

In [7]:
print(f"Data Shape before dropping columns: {df_train.shape}")
df_train = df_train.drop(
    columns=["lpep_pickup_datetime", "lpep_dropoff_datetime", "ehail_fee"]
)
df_train = df_train.dropna()

print(f"Data Shape after dropping columns: {df_train.shape}")

Data Shape before dropping columns: (41729, 23)
Data Shape after dropping columns: (36752, 20)


In [8]:
# Check for Duplicates
duplicates = df_train.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

print(f"Data Shape before dropping duplicates: {df_train.shape}")
df_train = df_train.drop_duplicates()
print(f"Data Shape after dropping duplicates: {df_train.shape}")

Number of duplicate rows: 12
Data Shape before dropping duplicates: (36752, 20)
Data Shape after dropping duplicates: (36740, 20)


In [9]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 36740 entries, 0 to 39148
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               36740 non-null  int32  
 1   store_and_fwd_flag     36740 non-null  str    
 2   RatecodeID             36740 non-null  float64
 3   PULocationID           36740 non-null  str    
 4   DOLocationID           36740 non-null  str    
 5   passenger_count        36740 non-null  float64
 6   trip_distance          36740 non-null  float64
 7   fare_amount            36740 non-null  float64
 8   extra                  36740 non-null  float64
 9   mta_tax                36740 non-null  float64
 10  tip_amount             36740 non-null  float64
 11  tolls_amount           36740 non-null  float64
 12  improvement_surcharge  36740 non-null  float64
 13  total_amount           36740 non-null  float64
 14  payment_type           36740 non-null  float64
 15  trip_type         

In [10]:
df_train["store_and_fwd_flag"].value_counts()

store_and_fwd_flag
N    36721
Y       19
Name: count, dtype: int64

In [11]:
# Define the target variable and features
train_dicts = df_train[categorical + numerical].to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)

### Prepare Validation Data

In [12]:
val_df = load_data("../data/raw/green_tripdata_2026-05.parquet")

val_df.info()

<class 'pandas.DataFrame'>
Index: 41729 entries, 0 to 44920
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               41729 non-null  int32         
 1   lpep_pickup_datetime   41729 non-null  datetime64[us]
 2   lpep_dropoff_datetime  41729 non-null  datetime64[us]
 3   store_and_fwd_flag     36752 non-null  str           
 4   RatecodeID             36752 non-null  float64       
 5   PULocationID           41729 non-null  str           
 6   DOLocationID           41729 non-null  str           
 7   passenger_count        36752 non-null  float64       
 8   trip_distance          41729 non-null  float64       
 9   fare_amount            41729 non-null  float64       
 10  extra                  41729 non-null  float64       
 11  mta_tax                41729 non-null  float64       
 12  tip_amount             41729 non-null  float64       
 13  tolls_amount     

In [13]:
val_df = val_df.drop(
    columns=["lpep_pickup_datetime", "lpep_dropoff_datetime", "ehail_fee"]
)
val_df = val_df.dropna()
val_df = val_df.drop_duplicates()

val_df.info()

<class 'pandas.DataFrame'>
Index: 36740 entries, 0 to 39148
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               36740 non-null  int32  
 1   store_and_fwd_flag     36740 non-null  str    
 2   RatecodeID             36740 non-null  float64
 3   PULocationID           36740 non-null  str    
 4   DOLocationID           36740 non-null  str    
 5   passenger_count        36740 non-null  float64
 6   trip_distance          36740 non-null  float64
 7   fare_amount            36740 non-null  float64
 8   extra                  36740 non-null  float64
 9   mta_tax                36740 non-null  float64
 10  tip_amount             36740 non-null  float64
 11  tolls_amount           36740 non-null  float64
 12  improvement_surcharge  36740 non-null  float64
 13  total_amount           36740 non-null  float64
 14  payment_type           36740 non-null  float64
 15  trip_type         

In [14]:
val_dicts = val_df[categorical + numerical].to_dict(orient="records")

X_val = dv.transform(val_dicts)

### Start Model Training

In [15]:
# Split Data into Train and Test Sets
target_column = "trip_duration"

y_train, y_val = df_train[target_column].values, val_df[target_column].values

print(f"Training Set Shape: {X_train.shape}, {y_train.shape}")
print(f"Test Set Shape: {X_val.shape}, {y_val.shape}")

Training Set Shape: (36740, 3211), (36740,)
Test Set Shape: (36740, 3211), (36740,)


In [16]:
# Train a LinearRegression model
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3211,)","[ 17.99,-17.19,-13.32,..., 25.58, 0. , 2.59]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,8.019
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3211
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3210)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](3211,)","[514.32,179.58, 41.49,..., 0.98, 0.54, 0. ]"


In [19]:
y_pred = lr.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("Linear Regression Performance:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R^2 Score: {r2:.2f}")

Linear Regression Performance:
RMSE: 4.44
MAE: 2.98
R^2 Score: 0.76


In [20]:
# Save the Model
os.makedirs("../models", exist_ok=True)
model_path = "../models/baseline.pkl"
with open(model_path, "wb") as f:
    pickle.dump(lr, f)

print(f"Model saved to {model_path}")

Model saved to ../models/baseline.pkl
